In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ExpenseMonitoring") \
    .getOrCreate()

data = [
    (1, "Food", 450, "2026-06-01"),
    (1, "Transport", 200, "2026-06-02"),
    (2, "Shopping", 1500, "2026-06-03"),
    (1, "Bills", 800, "2026-06-05"),
    (2, "Food", 350, "2026-06-06"),
    (3, "Entertainment", 2500, "2026-06-07"),
    (3, "Food", 300, "2026-06-08"),
    (2, "Bills", 950, "2026-06-09"),
    (1, "Shopping", 1800, "2026-06-10"),
    (3, "Transport", 150, "2026-06-11")
]

columns = [
    "user_id",
    "category",
    "amount",
    "expense_date"
]

df = spark.createDataFrame(data, columns)

df.show()

+-------+-------------+------+------------+
|user_id|     category|amount|expense_date|
+-------+-------------+------+------------+
|      1|         Food|   450|  2026-06-01|
|      1|    Transport|   200|  2026-06-02|
|      2|     Shopping|  1500|  2026-06-03|
|      1|        Bills|   800|  2026-06-05|
|      2|         Food|   350|  2026-06-06|
|      3|Entertainment|  2500|  2026-06-07|
|      3|         Food|   300|  2026-06-08|
|      2|        Bills|   950|  2026-06-09|
|      1|     Shopping|  1800|  2026-06-10|
|      3|    Transport|   150|  2026-06-11|
+-------+-------------+------+------------+



In [5]:
from pyspark.sql.functions import sum

monthly_spend = df.groupBy("user_id") \
                  .agg(sum("amount").alias("total_spend"))

monthly_spend.show()

+-------+-----------+
|user_id|total_spend|
+-------+-----------+
|      1|       3250|
|      2|       2800|
|      3|       2950|
+-------+-----------+



In [6]:
from pyspark.sql.functions import col

large_expenses = df.filter(col("amount") > 1000)

large_expenses.show()

+-------+-------------+------+------------+
|user_id|     category|amount|expense_date|
+-------+-------------+------+------------+
|      2|     Shopping|  1500|  2026-06-03|
|      3|Entertainment|  2500|  2026-06-07|
|      1|     Shopping|  1800|  2026-06-10|
+-------+-------------+------+------------+



In [7]:
from pyspark.sql.functions import avg

user_avg = df.groupBy("user_id") \
             .agg(avg("amount").alias("avg_amount"))

spikes = df.join(user_avg, "user_id") \
           .filter(col("amount") > col("avg_amount") * 2)

spikes.show()

+-------+-------------+------+------------+-----------------+
|user_id|     category|amount|expense_date|       avg_amount|
+-------+-------------+------+------------+-----------------+
|      1|     Shopping|  1800|  2026-06-10|            812.5|
|      3|Entertainment|  2500|  2026-06-07|983.3333333333334|
+-------+-------------+------+------------+-----------------+



In [8]:
monthly_spend.write \
    .mode("overwrite") \
    .csv("expense_summary")